---
## 5. Seguimiento de Pupilas y Ojos con MediaPipe 👁️

Para cosas super rápidas y precisas como los ojos, **MediaPipe Face Mesh** es el rey. Provee 468 puntos faciales en 3D.

Nos interesan los puntos alrededor de los ojos.
El iris tiene índices específicos. Vamos a pintar las pupilas.

In [1]:

import cv2
import numpy as np
import mediapipe as mp

# INDICES DE MEDIAPIPE PARA EL IRIS (Aproximados)
LEFT_IRIS = [474, 475, 476, 477]
RIGHT_IRIS = [469, 470, 471, 472]

mp_face_mesh = mp.solutions.face_mesh
captura = cv2.VideoCapture(0)

# Usamos refine_landmarks=True para obtener los puntos del iris
with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as face_mesh:

    while True:
        exito, imagen = captura.read()
        if not exito: break
        imagen = cv2.flip(imagen, 1)
        rgb_imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        
        resultados = face_mesh.process(rgb_imagen)
        
        if resultados.multi_face_landmarks:
            mesh_points = np.array([np.multiply([p.x, p.y], [imagen.shape[1], imagen.shape[0]]).astype(int) for p in resultados.multi_face_landmarks[0].landmark])
            
            # Dibujar Iris Izquierdo
            # (cv2.minEnclosingCircle nos da el centro y radio aproximado del iris)
            (c_x, c_y), radio = cv2.minEnclosingCircle(mesh_points[LEFT_IRIS])
            centro_izq = (int(c_x), int(c_y))
            cv2.circle(imagen, centro_izq, int(radio), (0, 255, 0), 1)
            cv2.circle(imagen, centro_izq, 1, (0, 0, 255), -1) # Pupila
            
            # Dibujar Iris Derecho
            (c_x, c_y), radio = cv2.minEnclosingCircle(mesh_points[RIGHT_IRIS])
            centro_der = (int(c_x), int(c_y))
            cv2.circle(imagen, centro_der, int(radio), (0, 255, 0), 1)
            cv2.circle(imagen, centro_der, 1, (0, 0, 255), -1)

        cv2.imshow('Seguimiento de Pupilas', imagen)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

captura.release()
cv2.destroyAllWindows()

---
## 6. Bonus: Máscara Facial (Snapchat Style) 🎭

Ya que tenemos los puntos de la cara con MediaPipe, podemos pintar sobre ellos para crear una máscara simple.

In [2]:
import mediapipe as mp
import cv2

mp_face_mesh = mp.solutions.face_mesh
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles

captura = cv2.VideoCapture(0)

with mp_face_mesh.FaceMesh(
    max_num_faces=1,
    refine_landmarks=True,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5) as face_mesh:

    while True:
        exito, imagen = captura.read()
        if not exito: break
        imagen = cv2.flip(imagen, 1)
        rgb_imagen = cv2.cvtColor(imagen, cv2.COLOR_BGR2RGB)
        
        resultados = face_mesh.process(rgb_imagen)
        
        if resultados.multi_face_landmarks:
            for face_landmarks in resultados.multi_face_landmarks:
                # Dibujamos la malla teselada (Tesselation) con estilo por defecto
                mp_drawing.draw_landmarks(
                    image=imagen,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_TESSELATION,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_tesselation_style())
                
                # Dibujamos contornos
                mp_drawing.draw_landmarks(
                    image=imagen,
                    landmark_list=face_landmarks,
                    connections=mp_face_mesh.FACEMESH_CONTOURS,
                    landmark_drawing_spec=None,
                    connection_drawing_spec=mp_drawing_styles.get_default_face_mesh_contours_style())

        cv2.imshow('Mascara Facial', imagen)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

captura.release()
cv2.destroyAllWindows()